# MySQL Contact → OneBill Migration

This notebook migrates **contact records** from a MySQL database into OneBill, attaching each contact to its corresponding OneBill subscriber account.

The migration:
1. Queries all Consumer contacts from MySQL (filtered by `AccountSegment` and non-null `FirstName`)
2. Fetches the OneBill subscriber ID for **each unique `AccountCode`** found in the contact data
3. Builds a contact payload for each row and PUTs it to the OneBill Subscriber API
4. Runs requests concurrently using a thread pool for throughput
5. Logs every outcome and produces a CSV of any failures

> **Before running in production:** remove or increase the `df.head(1)` sampling limit in the *Get MySQL Data* cell.

In [13]:
# %pip install mysql-connector-python
# %pip install sqlalchemy
# %pip install python-dotenv
import json
import pandas as pd
from sqlalchemy import create_engine
import requests
import logging
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import threading
import os
from collections import defaultdict

from dotenv import load_dotenv
load_dotenv(override=True)  # override=True ensures .env values take precedence


2026-05-11 09:52:25,064 [WARNING] python-dotenv could not parse statement starting at line 1
2026-05-11 09:52:25,066 [WARNING] python-dotenv could not parse statement starting at line 5
2026-05-11 09:52:25,068 [WARNING] python-dotenv could not parse statement starting at line 11


True

## Configuration

Set concurrency and token-refresh parameters, then build the database connection string and OneBill base URLs from environment variables (loaded from `.env` via `python-dotenv`).

In [14]:
MAX_WORKERS = 20
TOKEN_TTL_SECONDS = 3500  # Refresh token 100s before expiry (typical OAuth TTL is 3600s)

db_url = f'mysql+mysqlconnector://{os.environ["DB_USERNAME"]}:{os.environ["DB_PASSWORD"]}@{os.environ["DB_HOST"]}/bi_curated_views'
baseUrl = 'https://sandbox-sg.onebillsoftware.com'
accessTokenURL = f'{baseUrl}/oauth/token'


## Step 1 — Load Contact Data from MySQL

Query the `dynamics_contact` table, joining to `reporting_account` so we can filter to **Consumer** segment accounts only. Contacts without a `FirstName` are excluded as they cannot form a valid OneBill payload.

The result is a DataFrame where each row is one contact. The distinct set of `AccountCode` values drives the subsequent account-lookup loop.

In [15]:
query = """
SELECT 
    contact.* 
FROM 
    bi_curated_views.dynamics_contact contact
LEFT JOIN 
    bi_curated_views.reporting_account account
ON 
    contact.`AccountCode` = account.`AccountCode`
WHERE 
    account.`AccountSegment` = 'Consumer'
AND 
    contact.`FirstName` IS NOT NULL
ORDER BY 
    contact.`AccountCode`;
"""

engine = create_engine(db_url)
df = pd.read_sql(query, con=engine)
df = df.head(10)   # <-- REMOVE this line in production; limits to 1 row for testing

# Collect every unique AccountCode present in the contact data.
# The migration loop will iterate over this set to look up each
# subscriber's OneBill ID before attaching contacts.
all_account_codes = df['AccountCode'].dropna().unique().tolist()

print(f'Loaded {len(df):,} contact rows from MySQL')
print(f'Unique AccountCodes to process: {len(all_account_codes):,}')


Loaded 10 contact rows from MySQL
Unique AccountCodes to process: 10


## Step 2 — Logging

Configure a timestamped log file alongside console output. Every contact result (success or failure) is written here, giving a full audit trail for each migration run.

In [16]:
log_filename = f'contact_migration_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


## Step 3 — Thread-Safe OAuth Token Manager

OneBill uses OAuth 2.0 bearer tokens that expire after ~1 hour. With 20 concurrent workers each making hundreds of API calls, fetching a new token on every request would hammer the auth endpoint and create a bottleneck.

`TokenManager` solves this by caching the token in memory and proactively refreshing it 100 seconds before expiry. A `threading.Lock` ensures only one thread refreshes at a time — all others wait briefly, then reuse the newly cached token.

In [17]:
class TokenManager:
    """Thread-safe bearer token cache with proactive refresh."""

    def __init__(self):
        self._lock = threading.Lock()
        self._token: str | None = None
        self._expires_at: datetime = datetime.min

    def get_token(self) -> str:
        with self._lock:           # only one thread refreshes at a time
            if datetime.now() >= self._expires_at:
                self._refresh()    # others wait at the lock, then see valid token
            return self._token

    def _refresh(self):
        logger.info('Refreshing OAuth token...')
        token_data = {
            'grant_type':    'password',
            'client_id':     os.environ['CLIENT_ID'],
            'client_secret': os.environ['CLIENT_SECRET'],
            'username':      os.environ['API_USERNAME'],
            'password':      os.environ['API_PASSWORD'],
        }
        response = requests.post(
            accessTokenURL,
            data=token_data,
            headers={'Content-Type': 'application/x-www-form-urlencoded'}
        )
        response.raise_for_status()
        payload = response.json()
        self._token = payload['access_token']
        ttl = payload.get('expires_in', TOKEN_TTL_SECONDS)
        self._expires_at = datetime.now() + timedelta(seconds=ttl - 100)  # 100s safety margin
        logger.info('Token refreshed; valid until %s', self._expires_at.strftime('%H:%M:%S'))

token_manager = TokenManager()


## Step 4 — Contact Payload Builder

Transforms a single DataFrame row into the JSON body expected by the OneBill Subscriber Contact API. Handles `None` values and date serialisation so the API never receives a raw `NaT` or `NaN`.

In [18]:
def build_account_payload(row: pd.Series) -> str:
    """Map a DataFrame row to the OneBill Account API payload."""
    row = row.where(pd.notna(row), None).to_dict()

    def serialize_date(value, fmt=None):
        if value is None:
            return None
        if hasattr(value, 'isoformat'):
            return value.strftime(fmt) if fmt else value.isoformat()
        if fmt:
            try:
                return datetime.strptime(str(value), '%Y-%m-%d').strftime(fmt)
            except ValueError:
                return str(value)
        return str(value)

    return json.dumps({
        'contact': [
            {
                'id': row['ContactCode'],
                'firstName': row['FirstName'],
                'lastName':  row['LastName'],
                'ContactType': '1001',
                'primaryContact': 'false',
                'billingContact': 'false',
                'communicationPoint': [
                    {'type': 'Email', 'value': row['EmailAddresses']},
                    {'type': 'Phone', 'value': row['PhoneMobile']},
                    {'type': 'CPhone', 'value': row['PhoneHome']}
                ]   
            }
        ]
    })


## Step 5 — OneBill API Request

Sends a single `PUT` request to add a contact to a specific OneBill subscriber. The subscriber ID (`onebill_id`) is resolved per `AccountCode` before this function is called, so each request targets the correct account. The bearer token is fetched from `TokenManager` — already cached unless it is about to expire.

In [ ]:
def create_onebill_account(session: requests.Session, base_url: str, account_id: str, payload: str) -> dict:
    """PUT a contact onto a specific OneBill subscriber account.

    Args:
        session:    Shared requests.Session (connection-pooled).
        base_url:   OneBill environment root URL.
        account_id: OneBill subscriber ID resolved from AccountCode.
        payload:    JSON string built by build_account_payload().

    Returns:
        Parsed JSON response dict from OneBill.
    """
    # Build the endpoint URL using the resolved OneBill subscriber ID
    url = f'{base_url}/rest/SubscriberService/v1/subscribers/{account_id}'

    # Fetch the cached (or freshly refreshed) bearer token for this request
    headers = {
        'Authorization': f'Bearer {token_manager.get_token()}'
    }

    response = session.put(url, headers=headers, data=payload, timeout=30)
    response.raise_for_status()   # raise HTTPError for 4xx / 5xx responses
    data = response.json()

    # OneBill may return HTTP 200 but still signal validation failures in the body
    validation = data.get('validationResponse', {})
    if not validation.get('successful', True):
        errors   = validation.get('validationErrorInfo', [])
        messages = '; '.join(e.get('message', '') for e in errors)
        raise ValueError(messages)

    return data


## Step 6 — Per-Row Worker Function

Called once per contact row inside the thread pool. Builds the payload, resolves the OneBill subscriber ID for the row's `AccountCode`, calls the API, and returns a result dict. Timing is split into *build* (CPU) vs *network* (I/O) so the profiling summary can identify where time is actually spent.

In [20]:
def migrate_row(row: pd.Series, session: requests.Session, onebill_id_map: dict) -> dict:
    """Migrate a single contact row to OneBill.

    Args:
        row:            One row from the contacts DataFrame.
        session:        Shared HTTP session.
        onebill_id_map: Mapping of AccountCode → OneBill subscriber ID,
                        pre-built before the thread pool starts.
    """
    account_code = row['AccountCode']
    first_name   = row['FirstName']
    last_name    = row['LastName']
    contact_code  = row['ContactCode']

    # Look up the OneBill subscriber ID for this account code.
    # Rows whose AccountCode has no matching OneBill account are skipped.
    onebill_id = onebill_id_map.get(account_code)
    if not onebill_id:
        logger.warning(f'  [SKIP] {account_code} — no OneBill subscriber ID found')
        return {
            'account_code':     account_code,
            'contact_code':     contact_code,
            'first_name':       first_name,
            'last_name':        last_name,
            'onebill_id':       None,
            'status':           'skipped',
            'error':            'No OneBill subscriber ID found for this AccountCode',
            'elapsed_build_ms': 0,
            'elapsed_net_ms':   0,
        }

    # --- Time the payload build step separately from the network call ---
    t0      = time.perf_counter()
    payload = build_account_payload(row)
    t_build = time.perf_counter() - t0

    try:
        t1       = time.perf_counter()
        # Pass the resolved subscriber ID so the PUT hits the correct account
        response = create_onebill_account(session, baseUrl, onebill_id, payload)
        t_net    = time.perf_counter() - t1

        onebill_id_returned = response.get('accountId', 'unknown')
        logger.info(
            f'  [OK] {account_code} (ob={onebill_id}) '
            f'— build={t_build*1000:.0f}ms  net={t_net*1000:.0f}ms'
        )
        return {
            'account_code':     account_code,
            'contact_code':     contact_code,
            'first_name':       first_name,
            'last_name':        last_name,
            'onebill_id':       onebill_id,
            'status':           'success',
            'error':            None,
            'elapsed_build_ms': round(t_build * 1000, 1),
            'elapsed_net_ms':   round(t_net   * 1000, 1),
        }

    except Exception as e:
        t_net = time.perf_counter() - t1 if 't1' in dir() else 0
        logger.error(f'  [FAIL] {account_code} (ob={onebill_id}) — {e}')
        return {
            'account_code':     account_code,
            'first_name':       first_name,
            'last_name':        last_name,
            'onebill_id':       onebill_id,
            'status':           'failed',
            'error':            str(e),
            'elapsed_build_ms': round(t_build * 1000, 1),
            'elapsed_net_ms':   round(t_net   * 1000, 1),
        }


## Step 7 — Account-Code Loop & Migration Orchestrator

The `build_onebill_id_map()` function iterates over every unique `AccountCode` in the dataset and queries OneBill for the corresponding subscriber ID. This lookup happens **before** the thread pool starts, so all workers share a pre-built dictionary — avoiding redundant API calls and race conditions.

`migrate()` then fans the contact rows out across a `ThreadPoolExecutor`, collecting results and logging progress every 50 rows. A profiling summary is printed on completion.

In [ ]:
def get_onebill_subscriber_id(session: requests.Session, base_url: str, account_code: str):
    """Look up the OneBill subscriber ID for a given AccountCode.

    Returns the subscriber ID string, or None if not found / on error.
    """
    url = f'{base_url}/rest/SubscriberService/v1/subscribers'
    headers = {'Authorization': f'Bearer {token_manager.get_token()}'}

    try:
        # Search OneBill for a subscriber matching this account code
        response = session.get(
            url,
            params={'accountCode': account_code},
            timeout=30
        )
        response.raise_for_status()
        data = response.json()

        # OneBill returns a list; take the first match's ID
        subscribers = data.get('subscribers', [])
        if subscribers:
            return str(subscribers[0].get('id'))
        return None

    except Exception as e:
        logger.warning(f'  [LOOKUP FAIL] {account_code} — {e}')
        return None


def build_onebill_id_map(
    session: requests.Session,
    account_codes: list,
    max_workers: int = MAX_WORKERS
) -> dict:
    """Resolve OneBill subscriber IDs for all AccountCodes before migration begins.

    Runs lookups concurrently, then returns a dict of {AccountCode: onebill_id}.
    AccountCodes with no match are included with a None value so migrate_row
    can detect and skip them cleanly.
    """
    logger.info(f'Resolving OneBill IDs for {len(account_codes):,} unique AccountCodes...')
    id_map: dict = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit one lookup future per unique AccountCode
        futures = {
            executor.submit(get_onebill_subscriber_id, session, baseUrl, code): code
            for code in account_codes
        }
        for future in as_completed(futures):
            code          = futures[future]
            id_map[code]  = future.result()   # None if lookup failed

    found   = sum(1 for v in id_map.values() if v is not None)
    missing = len(id_map) - found
    logger.info(f'ID resolution complete — {found} found, {missing} not matched in OneBill')
    return id_map


def migrate(df: pd.DataFrame, max_workers: int = MAX_WORKERS) -> pd.DataFrame:
    """Migrate every contact row in df to OneBill, returning a results DataFrame."""

    # Build a shared, connection-pooled HTTP session for all workers
    session = requests.Session()
    adapter = requests.adapters.HTTPAdapter(
        pool_connections=max_workers,
        pool_maxsize=max_workers        # keep enough connections for all threads
    )
    session.mount('https://', adapter)
    session.headers.update({
        'proxy_accountNumber': os.environ['PROXY_ACCOUNT_NUMBER'] # Partner/proxy account — scopes all requests to your accounts in OneBill
        'Content-Type': 'application/json',
    })

    # ── Phase 1: resolve all AccountCodes to OneBill subscriber IDs ──
    # Done once, upfront, so workers share an immutable lookup dict
    unique_codes  = df['AccountCode'].dropna().unique().tolist()
    onebill_id_map = build_onebill_id_map(session, unique_codes, max_workers)

    # ── Phase 2: fan contact rows out across the thread pool ──────────
    rows  = [row for _, row in df.iterrows()]
    total = len(rows)
    results: list[dict] = []

    logger.info(f'Starting contact migration of {total:,} rows with {max_workers} workers...')
    wall_start = time.perf_counter()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Each future carries the row, shared session, and pre-built ID map
        futures = {
            executor.submit(migrate_row, row, session, onebill_id_map): row['AccountCode']
            for row in rows
        }

        for i, future in enumerate(as_completed(futures), start=1):
            result = future.result()
            results.append(result)

            # Log a progress summary every 50 rows and on the final row
            if i % 50 == 0 or i == total:
                ok      = sum(1 for r in results if r['status'] == 'success')
                skipped = sum(1 for r in results if r['status'] == 'skipped')
                fail    = sum(1 for r in results if r['status'] == 'failed')
                logger.info(f'Progress: {i}/{total} — {ok} ok, {skipped} skipped, {fail} failed')

    wall_elapsed = time.perf_counter() - wall_start

    results_df = pd.DataFrame(results)
    success = (results_df['status'] == 'success').sum()
    skipped = (results_df['status'] == 'skipped').sum()
    failed  = (results_df['status'] == 'failed').sum()

    logger.info(
        f'Migration done in {wall_elapsed:.1f}s — '
        f'{success} succeeded, {skipped} skipped, {failed} failed. (log: {log_filename})'
    )

    # ── Profiling summary (network bottleneck diagnosis) ──────────────
    net_times = results_df.loc[results_df['elapsed_net_ms'] > 0, 'elapsed_net_ms']
    print('\n=== Profiling Summary ===')
    print(f'Total wall time:          {wall_elapsed:.1f}s')
    print(f'Throughput:               {total / wall_elapsed:.1f} contacts/s')
    print(f'Avg build time per row:   {results_df["elapsed_build_ms"].mean():.1f}ms')
    if not net_times.empty:
        print(f'Avg network time per row: {net_times.mean():.1f}ms')
        print(f'Max network time:         {net_times.max():.1f}ms')
        print(f'P95 network time:         {net_times.quantile(0.95):.1f}ms')
    print('=========================')

    return results_df


## Step 8 — Run the Migration

Executes the full pipeline:
1. Resolves OneBill subscriber IDs for every unique `AccountCode` in the dataset
2. Migrates all contact rows concurrently
3. Prints a profiling summary and displays any failed rows inline

Failed rows are also exported to `Failed_Contact_Migrations.csv` for review and retry.

In [22]:
# Run the full migration pipeline
results_df = migrate(df)

# Separate outcomes for review
failures = results_df[results_df['status'] == 'failed']
skipped  = results_df[results_df['status'] == 'skipped']

print(f'\nFailed rows  ({len(failures)}):')
display(failures)

print(f'\nSkipped rows ({len(skipped)}) — no OneBill subscriber ID found:')
display(skipped)


2026-05-11 09:52:41,586 [INFO] Resolving OneBill IDs for 10 unique AccountCodes...
2026-05-11 09:52:41,593 [INFO] Refreshing OAuth token...
2026-05-11 09:52:43,414 [INFO] Token refreshed; valid until 10:45:55
2026-05-11 09:53:08,630 [INFO] ID resolution complete — 0 found, 10 not matched in OneBill
2026-05-11 09:53:08,633 [INFO] Starting contact migration of 10 rows with 20 workers...
2026-05-11 09:53:08,635 [WARNING]   [SKIP] 10625002 — no OneBill subscriber ID found
2026-05-11 09:53:08,637 [WARNING]   [SKIP] 12579173 — no OneBill subscriber ID found
2026-05-11 09:53:08,639 [WARNING]   [SKIP] 16673533 — no OneBill subscriber ID found
2026-05-11 09:53:08,641 [WARNING]   [SKIP] 21186140 — no OneBill subscriber ID found
2026-05-11 09:53:08,641 [WARNING]   [SKIP] 24000898 — no OneBill subscriber ID found
2026-05-11 09:53:08,642 [WARNING]   [SKIP] 25262934 — no OneBill subscriber ID found
2026-05-11 09:53:08,644 [WARNING]   [SKIP] 25342741 — no OneBill subscriber ID found
2026-05-11 09:53:


=== Profiling Summary ===
Total wall time:          0.0s
Throughput:               275.4 contacts/s
Avg build time per row:   0.0ms

Failed rows  (0):


,account_code,contact_code,first_name,last_name,onebill_id,status,error,elapsed_build_ms,elapsed_net_ms



Skipped rows (10) — no OneBill subscriber ID found:


,account_code,contact_code,first_name,last_name,onebill_id,status,error,elapsed_build_ms,elapsed_net_ms
0,10625002,BILLING-10625002,Anna,Milroy,None,skipped,No OneBill subscriber ID found for this Accoun...,0,0
1,12579173,BILLING-12579173,Cara Tipping Smith t/as Copy Carats,,None,skipped,No OneBill subscriber ID found for this Accoun...,0,0
2,16673533,BILLING-16673533,Jason,Yee,None,skipped,No OneBill subscriber ID found for this Accoun...,0,0
3,21186140,BILLING-21186140,Lyn-Marie,Harris,None,skipped,No OneBill subscriber ID found for this Accoun...,0,0
4,24000898,BILLING-24000898,Kylie,Glenn,None,skipped,No OneBill subscriber ID found for this Accoun...,0,0
5,25262934,BILLING-25262934,Rata,Miller,None,skipped,No OneBill subscriber ID found for this Accoun...,0,0
6,25342741,BILLING-25342741,Cliff,Black,None,skipped,No OneBill subscriber ID found for this Accoun...,0,0
7,25928300,BILLING-25928300,Jon,Verhoek,None,skipped,No OneBill subscriber ID found for this Accoun...,0,0
8,26302594,BILLING-26302594,Nathan,Kennedy,None,skipped,No OneBill subscriber ID found for this Accoun...,0,0
9,27386946,BILLING-27386946,Dana,Waitai-Cross,None,skipped,No OneBill subscriber ID found for this Accoun...,0,0


In [23]:
# Export all non-successful rows to CSV for manual review / retry
non_success = results_df[results_df['status'] != 'success']
non_success.to_csv('Failed_Contact_Migrations.csv', index=False)
print(f'Exported {len(non_success):,} non-successful rows to Failed_Contact_Migrations.csv')


Exported 10 non-successful rows to Failed_Contact_Migrations.csv
